# Data Preparation — Saudi Real Estate Market
This notebook prepares and cleans raw data for the Saudi real estate market analysis.

The notebook performs the following tasks:
- Load raw rental and sales datasets from `raw_dataset`
- Clean and standardize the sales and rental data
- Map Arabic city names to English city names
- Filter to target cities and property types
- Generate a merged `master_real_estate.csv` table
- Create supporting datasets: average income, population, interest rate, and inflation

Notes:
- Run cells from top to bottom.
- Verify `raw_dataset` contains the expected source files before running.

## 1. Setup and environment
Load the required libraries and ensure the output folder exists.

In [ ]:
# Import required Python libraries for data processing and path handling
import pandas as pd
import numpy as np
from pathlib import Path

# Configure pandas display settings for easier debugging and previewing
pd.options.display.max_columns = 200

# Create the output folder if it does not already exist
raw_dir = Path('raw_dataset')
cleaned_dir = Path('cleaned_dataset')
cleaned_dir.mkdir(parents=True, exist_ok=True)

print('raw_dataset exists:', raw_dir.exists())
print('cleaned_dataset exists:', cleaned_dir.exists())

## 2. Constants and helper functions
Define common mappings and helper functions used across the notebook.

In [ ]:
# Define the Arabic city names expected in the raw input files
target_cities_ar = [
    'الرياض', 'جدة', 'مكة المكرمة', 'مكه المكرمه', 'المدينة المنورة', 'المدينه المنوره',
    'الدمام', 'الخبر', 'أبها', 'ابها', 'جازان', 'نجران',
    'تبوك', 'حائل', 'بريدة', 'سكاكا', 'عرعر', 'الباحة',"بريده"
]

# Map Arabic city names to English city names used in cleaned output
city_mapping = {
    'الرياض': 'Riyadh',
    'جدة': 'Jeddah',
    'مكة المكرمة': 'Makkah',
    'مكه المكرمه': 'Makkah',
    'المدينة المنورة': 'Madinah',
    'المدينه المنوره': 'Madinah',
    'الدمام': 'Dammam',
    'الخبر': 'Khobar',
    'أبها': 'Abha',
    'ابها': 'Abha',
    'جازان': 'Jazan',
    'نجران': 'Najran',
    'تبوك': 'Tabuk',
    'حائل': 'Hail',
    'بريدة': 'Buraidah',
    'سكاكا': 'Sakaka',
    'عرعر': 'Arar',
    'الباحة': 'Al Bahah',
    "بريده": "Buraidah"
}

# Map English city names to Arabic regions for region-level analysis
city_region_map = {
    'Riyadh': 'منطقة الرياض',
    'Jeddah': 'منطقة مكة المكرمة',
    'Makkah': 'منطقة مكة المكرمة',
    'Madinah': 'منطقة المدينة المنورة',
    'Dammam': 'المنطقة الشرقية',
    'Khobar': 'المنطقة الشرقية',
    'Abha': 'منطقة عسير',
    'Jazan': 'منطقة جازان',
    'Najran': 'منطقة نجران',
    'Tabuk': 'منطقة تبوك',
    'Hail': 'منطقة حائل',
    'Buraidah': 'منطقة القصيم',
    'Sakaka': 'منطقة الجوف',
    'Arar': 'منطقة الحدود الشمالية',
    'Al Bahah': 'منطقة الباحة'
}

# Define helper functions used across multiple cleaning steps
def make_quarter_key(df, year_col='year', quarter_col='quarter'):
    return df[year_col].astype(str) + 'Q' + df[quarter_col].astype(str)

def classify_sales_property(value):
    text = str(value)
    if any(term in text for term in ['سكنى', 'فيلا', 'شقة', 'عمارة']):
        return 'Residential'
    if any(term in text for term in ['تجارى', 'تجاري', 'زراعي']):
        return 'Commercial'
    return 'Other'

def classify_rent_property(value):
    text = str(value)
    if any(term in text for term in ['سكني', 'سكنى']):
        return 'Residential'
    if any(term in text for term in ['تجاري', 'تجارى']):
        return 'Commercial'
    return 'Other'

def safe_load_table(path):
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    return pd.read_excel(path)

## 3. Load rental source files
Load all regional rental files and combine them before cleaning.

In [ ]:
# List all raw rental source files that should exist in raw_dataset
rental_files = [
    'Rental indicators for Cities in Riyadh region E.xlsx',
    'Rental indicators for Cities in Al Baha region E.xlsx',
    'Rental indicators for Cities in Makkah region.csv',
    'Rental indicators for Cities in Madinah region.csv',
    'Rental indicators for Cities in Qassim region E.xlsx',
    'Rental indicators for Cities in Tabuk regionE.xlsx',
    'Rental indicators for Cities in Hail regionE.xlsx',
    'Rental indicators for Cities in N B region E.xlsx',
    'Rental indicators for Cities in Jazan regionE.xlsx',
    'Rental indicators for Cities Asir region E.xlsx',
    'Rental indicators for Cities in Eastern Province E.xlsx',
    'Rental indicators for Cities in Al jawf regionE.xlsx',
    'Rental indicators for Cities in Najran regionE.xlsx',
]

# Load each rental file if available, and keep the filename for traceability
rental_dfs = []
for file_name in rental_files:
    file_path = raw_dir / file_name
    if not file_path.exists():
        print('Warning: missing rental file', file_name)
        continue
    df = safe_load_table(file_path)
    df.columns = df.columns.str.strip()
    df['source_file'] = file_name
    rental_dfs.append(df)

# Normalize column names and combine regional tables into one dataframe
print('Loaded rental tables:', len(rental_dfs))
df_rent_raw = pd.concat(rental_dfs, ignore_index=True) if rental_dfs else pd.DataFrame()
print('Combined rental shape:', df_rent_raw.shape)
display(df_rent_raw.head())


## 4. Clean rental data
Standardize column names, filter target cities, map city names to English, and classify property types.

In [ ]:
# Standardize rental dataset schema and validate required columns
rent_rename = {
    'السنة': 'year',
    'السنة ': 'year',
    'الربع': 'quarter',
    'المنطقة': 'region_ar',
    'المنطقة ': 'region_ar',
    'المدينة': 'city_ar',
    'المدينة ': 'city_ar',
    'نوع العقار': 'property_category_ar',
    'نوع العقار ': 'property_category_ar',
    'مجموع الصفقات': 'rental_contracts',
    'المتوسط': 'avg_rent'
}

# Filter rental rows to the selected cities and translate city names
if df_rent_raw.empty:
    raise ValueError('No rental data loaded. Check raw_dataset file names.')

df_rent = df_rent_raw.rename(columns=rent_rename).copy()
required = ['year', 'quarter', 'city_ar', 'property_category_ar', 'rental_contracts', 'avg_rent']
missing = [col for col in required if col not in df_rent.columns]
if missing:
    raise ValueError('Missing rental columns: ' + str(missing))

df_rent = df_rent[df_rent['city_ar'].isin(target_cities_ar)].copy()
df_rent['city'] = df_rent['city_ar'].map(city_mapping)

# Classify rental property categories into broad labels
df_rent['property_type'] = df_rent['property_category_ar'].apply(classify_rent_property)
df_rent = df_rent[df_rent['property_type'].isin(['Residential', 'Commercial'])].copy()

# Generate a quarter key for later merging with sales data
df_rent['quarter_key'] = make_quarter_key(df_rent)
df_rent = df_rent[[
    'quarter_key', 'year', 'quarter', 'region_ar', 'city',
    'property_type', 'rental_contracts', 'avg_rent', 'property_category_ar'
]]

# Save the cleaned rental dataset for future use
df_rent.to_csv(cleaned_dir / 'rent_cleaned.csv', index=False, encoding='utf-8-sig')
print('Cleaned rent saved:', df_rent.shape)
display(df_rent.head())


## 5. Load and clean sales data
Select relevant sales columns, apply city mapping, and create the cleaned sales dataset.

In [ ]:
# Load the raw sales dataset and clean column names
sales_path = raw_dir / 'quarter_report SI.xlsx'
df_sales_raw = safe_load_table(sales_path)
df_sales_raw.columns = df_sales_raw.columns.str.strip()

# Keep only the relevant columns needed for analysis
sales_columns = [
    'yearnumber', 'quarternumber', 'region_ar', 'city_ar',
    'typecategoryar', 'deed_counts', 'RealEstatePrice_SUM', 'Meter_Price_W_Avg_IQR'
]
df_sales = df_sales_raw[sales_columns].copy()
df_sales = df_sales.rename(columns={
    'yearnumber': 'year',
    'quarternumber': 'quarter',
    'typecategoryar': 'property_category_ar',
    'deed_counts': 'sales_transactions',
    'RealEstatePrice_SUM': 'market_value',
    'Meter_Price_W_Avg_IQR': 'avg_price_m2'
})

# Filter sales data to target cities and map city names to English
df_sales = df_sales[df_sales['city_ar'].isin(target_cities_ar)].copy()
df_sales['city'] = df_sales['city_ar'].map(city_mapping)

# Classify sales property categories into broad labels
df_sales['property_type'] = df_sales['property_category_ar'].apply(classify_sales_property)
df_sales = df_sales[df_sales['property_type'].isin(['Residential', 'Commercial'])].copy()
df_sales['quarter_key'] = make_quarter_key(df_sales)
df_sales = df_sales[[
    'quarter_key', 'year', 'quarter', 'region_ar', 'city',
    'property_type', 'sales_transactions', 'market_value', 'avg_price_m2', 'property_category_ar'
]]

# Save the cleaned sales dataset for future use
df_sales.to_csv(cleaned_dir / 'sales_cleaned.csv', index=False, encoding='utf-8-sig')
print('Cleaned sales saved:', df_sales.shape)
display(df_sales.head())

## 6. Build the master dataset
Aggregate sales and rental data by quarter, city, region, and property type.

In [ ]:
# Load cleaned sales and rental datasets from disk
sales = pd.read_csv(cleaned_dir / 'sales_cleaned.csv')
rent = pd.read_csv(cleaned_dir / 'rent_cleaned.csv')

# Define the merge grain for the final master dataset
merge_keys = ['quarter_key', 'year', 'quarter', 'region_ar', 'city', 'property_type']


# Aggregate sales and rental values with weighted averages
sales_work = sales.copy()
sales_work['weighted_price_m2'] = (
    sales_work['avg_price_m2'] * sales_work['sales_transactions']
).where(sales_work['avg_price_m2'].notna(), 0)
sales_work['price_weight'] = sales_work['sales_transactions'].where(sales_work['avg_price_m2'].notna(), 0)

sales_agg = (
    sales_work.groupby(merge_keys, as_index=False, dropna=False)
    .agg(
        sales_transactions=('sales_transactions', 'sum'),
        market_value=('market_value', 'sum'),
        weighted_price_m2=('weighted_price_m2', 'sum'),
        price_weight=('price_weight', 'sum'),
    )
)
sales_agg['avg_price_m2'] = np.where(
    sales_agg['price_weight'] > 0,
    sales_agg['weighted_price_m2'] / sales_agg['price_weight'],
    np.nan
)
sales_agg = sales_agg.drop(columns=['weighted_price_m2', 'price_weight'])

rent_work = rent.copy()
rent_work['weighted_rent'] = (
    rent_work['avg_rent'] * rent_work['rental_contracts']
).where(rent_work['avg_rent'].notna(), 0)
rent_work['rent_weight'] = rent_work['rental_contracts'].where(rent_work['avg_rent'].notna(), 0)

rent_agg = (
    rent_work.groupby(merge_keys, as_index=False, dropna=False)
    .agg(
        rental_contracts=('rental_contracts', 'sum'),
        weighted_rent=('weighted_rent', 'sum'),
        rent_weight=('rent_weight', 'sum'),
    )
)
rent_agg['avg_rent'] = np.where(
    rent_agg['rent_weight'] > 0,
    rent_agg['weighted_rent'] / rent_agg['rent_weight'],
    np.nan
)
rent_agg = rent_agg.drop(columns=['weighted_rent', 'rent_weight'])

print('Sales aggregate rows:', sales_agg.shape[0])
print('Rent aggregate rows:', rent_agg.shape[0])

assert not sales_agg.duplicated(subset=merge_keys).any(), 'Duplicate sales merge keys'
assert not rent_agg.duplicated(subset=merge_keys).any(), 'Duplicate rent merge keys'


# Merge the sales and rental aggregates into a unified master table
master = pd.merge(
    sales_agg,
    rent_agg,
    on=merge_keys,
    how='outer',
    validate='one_to_one'
)


# Fill any missing region labels and save the final master dataset
master['region_ar'] = master['region_ar'].fillna(master['city'].map(city_region_map))

master = master[[
    'quarter_key', 'year', 'quarter', 'region_ar', 'city', 'property_type',
    'sales_transactions', 'market_value', 'avg_price_m2', 'rental_contracts', 'avg_rent'
]]

duplicate_count = master.duplicated(subset=['city', 'quarter_key', 'property_type']).sum()
print('Duplicate master rows by city-quarter-property_type:', duplicate_count)

master.to_csv(cleaned_dir / 'master_real_estate.csv', index=False, encoding='utf-8-sig')
print('Master dataset saved:', master.shape)
display(master.head())

## 7. Create supporting datasets
Build auxiliary CSV files for income, population, interest rate, and inflation.

In [ ]:
# Create supporting datasets for income, population, interest rate, and inflation
# Load income data from the published survey and trim to the table section
income_path = raw_dir / 'Publication tables of Household Income and Consumption Expenditure Survey 2023_2.xlsx'
df_income = pd.read_excel(income_path, sheet_name='1-2')
df_income = df_income.iloc[4:17, 0:4].copy()
df_income = df_income[[ 'Unnamed: 0', 'الفهرس\ncontent' ]]
df_income
df_income = df_income.rename(columns={
    'Unnamed: 0': 'Region',
    'الفهرس\ncontent': 'Average_Income'
})
df_income['Region'] = df_income['Region'].astype(str).str.split('\n').str[-1].str.strip()
df_income.to_csv(cleaned_dir / 'average_income.csv', index=False, encoding='utf-8-sig')
print('Average income saved:', df_income.shape)
display(df_income.head())



# Build a small population lookup table for the target cities
population_data = {
    'city': ['Riyadh', 'Jeddah', 'Makkah', 'Madinah', 'Dammam', 'Khobar',
             'Abha', 'Jazan', 'Najran', 'Tabuk', 'Hail', 'Buraidah', 'Sakaka', 'Arar', 'Al Bahah'],
    'population': [7009120, 3751722, 2427924, 1477047, 1532326, 658550,
                   422243, 200911, 421902, 623665, 498575, 677647, 236669, 202719, 90515]
}
df_population = pd.DataFrame(population_data)
df_population.to_csv(cleaned_dir / 'population.csv', index=False, encoding='utf-8-sig')
print('Population saved:', df_population.shape)
display(df_population.head())

# Filter the interest rate data to the quarterly RR indicator
interest_path = raw_dir / 'interest-rates-and-sama-average-bills.xlsx'
df_interest = pd.read_excel(interest_path)
df_interest = df_interest[(df_interest['Periodicity'] == 'Quarterly') & (df_interest['Indicator'] == 'RR')].copy()
df_interest = df_interest[['quarter', 'Year', 'Interest Rate']].rename(columns={
    'Year': 'year',
    'Interest Rate': 'repo_rate'
})
df_interest['quarter_key'] = make_quarter_key(df_interest, year_col='year', quarter_col='quarter')
df_interest = df_interest[['quarter_key', 'year', 'quarter', 'repo_rate']]
df_interest.to_csv(cleaned_dir / 'interest_rate.csv', index=False, encoding='utf-8-sig')
print('Interest rate saved:', df_interest.shape)
display(df_interest.head())

# Save the static inflation series used by the analysis
inflation = pd.DataFrame({
    'year': [2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'inflation_rate': [2.46, -2.09, 3.45, 3.06, 2.47, 2.33, 1.69]
})
inflation.to_csv(cleaned_dir / 'inflation.csv', index=False, encoding='utf-8-sig')
print('Inflation saved:', inflation.shape)
display(inflation)

## 8. Summary
The following files are now ready for analysis:
- `sales_cleaned.csv`
- `rent_cleaned.csv`
- `master_real_estate.csv`
- `average_income.csv`
- `population.csv`
- `interest_rate.csv`
- `inflation.csv`